# Replica exchange

Replica exchange runs of 7ez0 and 7ez0_noion (μMg, 8 replicas per replicate): bias and λ time series, replica and λ occupancy, Mg²⁺ binding per replica and per λ.

## Prerequisites

Stages 5a and 6 of `PIPELINE.md`, for `replex/{7ez0,7ez0_noion}/mMg_tip3p/replicate{0,1}/`:

- `rep*/bias.COLVAR.<i>`, `rep*/bias_index.xvg`: PLUMED outputs of the runs (`replex/prepare.sh`, `replex/run.slurm`)
- `rep*/cv.COLVAR.<i>`: coordination numbers per Mg²⁺ (`replex/analysis.slurm`)
- `rep*/dist_MG_O_Phosphate.xvg`, `rep*/dist_O_Phosphate_MG.xvg`: minimum distances (`replex/compute_ion_dist.slurm`)

## Outputs

In `figures/replex/`, for each system and replicate (`*_{system}_mMg_tip3p_replicate{rep}.pdf`): `bias_replica`, `bias_lambda`, `bias_dist`, `bound_mg_replica`, `bound_mg_lambda`, `n_inner_dist`, `n_outer_dist`, `n_tot_dist`, `replica_occupancy`, `lambda_occupancy`, `heatmap_mg_index`, `bound_mg_denticity`. Only the first system and replicate are shown inline.

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

from mg_sampling.io import read_plumed
from mg_sampling.replex import demux
from mg_sampling.stats import edges_to_range
from pipeline import PROJECT_DIR
from pipeline.constants import RT

plt.style.use("paper.mplstyle")
os.makedirs("figures/replex", exist_ok=True)
np.seterr(divide="ignore")  # empty histogram bins: infinite free energy
# Restarts repeat rows in bias_index.xvg; demux drops them
logging.getLogger().addFilter(lambda record: "Found repeated times" not in record.getMessage())

In [ ]:
ff = 'mMg_tip3p'
nrep = 8  # replicas per replicate
PAIRS = [(system, rep) for system in ['7ez0', '7ez0_noion'] for rep in range(2)]


def load(system, rep):
    path = PROJECT_DIR / f'replex/{system}/{ff}/replicate{rep}'
    repindex_file = path / f'rep{nrep*rep:02d}/bias_index.xvg'

    cv_dmx = np.stack([
        read_plumed(path / f'rep{i+nrep*rep:02d}/bias.COLVAR.{i}').drop_duplicates('time').assign(rep=i)
        for i in range(nrep)
    ])
    cv = demux(cv_dmx, repindex_file=repindex_file)

    nox_dmx = np.stack([
        read_plumed(path / f'rep{i+nrep*rep:02d}/cv.COLVAR.{i}').drop_duplicates('time').assign(rep=i)
        for i in range(nrep)
    ])
    nox_dmx[:, :, 1:-2:3] *= 1.039661
    nox_dmx[:, :, 2:-2:3] *= 1.039661
    nox = demux(nox_dmx, repindex_file=repindex_file)

    d_mg_op_dmx, d_op_mg_dmx = [
        np.stack([
            np.insert(np.loadtxt(path / f'rep{i+nrep*rep:02d}/{name}.xvg', comments=['@', '#']), 1, i, axis=-1)
            for i in range(nrep)
        ])
        for name in ['dist_MG_O_Phosphate', 'dist_O_Phosphate_MG']
    ]
    d_mg_op = demux(d_mg_op_dmx, repindex_file=repindex_file)
    d_op_mg = demux(d_op_mg_dmx, repindex_file=repindex_file)
    return cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg


def save(figs, name, system, rep):
    # One PDF page per figure; only the first (system, replicate) stays inline
    filename = f'figures/replex/{name}_{system}_{ff}_replicate{rep}.pdf'
    if isinstance(figs, list):
        with PdfPages(filename) as pdf:
            for fig in figs:
                pdf.savefig(fig, bbox_inches='tight')
    else:
        figs.savefig(filename)
        figs = [figs]
    if (system, rep) != PAIRS[0]:
        for fig in figs:
            plt.close(fig)


data = {(system, rep): load(system, rep) for system, rep in PAIRS}

## Figures

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    figs = []
    for n in range(4):
        fig, axarr = plt.subplots(nrows=3, sharex=True)
        fig.suptitle(f"{system} {ff} replicate {rep} ({n})")
        for i in range(n, nrep, 4):
            ax = axarr[0]
            ax.plot(cv_dmx[i, :, 0]/1000, cv_dmx[i, :, 1], label=f'rep{i:02d}')
            ax.set_ylabel(r"$U_\mathrm{bias}$ [kJ/mol]")
            ax.legend()
            ax = axarr[1]
            ax.plot(cv_dmx[i, :, 0]/1000, cv_dmx[i, :, -1], label=f'rep{i:02d}')
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_ylabel("Replica index")
            ax.set_xlabel("Time [ns]")
            ax.legend()
            ax = axarr[2]
            ax.plot(cv_dmx[i, :, 0]/1000, cv_dmx[i, :, -2], label=f'rep{i:02d}')
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_ylabel("$λ$ index")
            ax.set_xlabel("Time [ns]")
            ax.legend()
        figs.append(fig)
    save(figs, 'bias_replica', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    figs = []
    for n in range(4):
        fig, axarr = plt.subplots(nrows=3, sharex=True)
        fig.suptitle(f"{system} {ff} replicate {rep} ({n})")
        for i in range(n, nrep, 4):
            ax = axarr[0]
            ax.plot(cv[i, :, 0]/1000, cv[i, :, 1], label=f'$λ_{{{i:02d}}}$')
            ax.set_ylabel(r"$U_\mathrm{bias}$ [kJ/mol]")
            ax.legend()
            ax = axarr[1]
            ax.plot(cv[i, :, 0]/1000, cv[i, :, -1], label=f'$λ_{{{i:02d}}}$')
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_ylabel("$λ$ index")
            ax.set_xlabel("Time [ns]")
            ax.legend()
            ax = axarr[2]
            ax.plot(cv[i, :, 0]/1000, cv[i, :, -2], label=f'$λ_{{{i:02d}}}$')
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_ylabel("Replica index")
            ax.set_xlabel("Time [ns]")
            ax.legend()
        figs.append(fig)
    save(figs, 'bias_lambda', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    # Bias distribution as a function of λ
    fig, ax = plt.subplots(figsize=(8, 4))
    for i in range(1, nrep):
        ρ_u, bins = np.histogram(cv[i, :, 1], bins=50, density=True)
        u_range = edges_to_range(bins)
        ax.plot(u_range, ρ_u, label=f'$λ_{{{i}}}$', color=plt.cm.coolwarm(i/nrep))
    ax.legend()
    ax.set_xlabel(r'$U_\mathrm{bias}$ [kJ/mol]')
    ax.set_ylabel(r'$ρ(U_\mathrm{bias}$)')
    fig.suptitle(f"{system} {ff} replicate {rep}")
    save(fig, 'bias_dist', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    fig, axarr = plt.subplots(nrows=3, ncols=4, sharex=True, sharey='row', figsize=(10, 4))
    for i in range(nrep):
        n = i % 4
        ax = axarr[0, n]
        ax.plot(d_mg_op_dmx[i, :, 0] / 1000, (d_mg_op_dmx[i, :, 2:] < 0.25).sum(axis=-1), label=f'rep{i:02d}', color=plt.cm.rainbow(i/(nrep-1)), lw=1)
        ax.set_ylabel('# Bound MG')
        ax.legend(loc='lower center', ncols=2, bbox_to_anchor=(0.5, 1), frameon=False)
        ax = axarr[1, n]
        ax.plot(d_op_mg_dmx[i, :, 0] / 1000, (d_op_mg_dmx[i, :, 2:] < 0.25).sum(axis=-1), label=f'rep{i:02d}', color=plt.cm.rainbow(i/(nrep-1)), lw=1)
        ax.set_ylabel('# Bound OP')
        ax = axarr[2, n]
        ax.plot(cv_dmx[i, ::10, 0] / 1000, cv_dmx[i, ::10, -2], label=f'rep{i:02d}', color=plt.cm.rainbow(i/(nrep-1)), lw=1)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_ylabel('λ index')
        ax.set_xlabel("Time [ns]")
        ax.set_xlim(0, cv_dmx[i, :, 0].max() / 1000)
    save(fig, 'bound_mg_replica', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    fig, axarr = plt.subplots(nrows=3, ncols=4, sharex=True, sharey='row', figsize=(10, 4))
    for n in range(4):
        for i in range(n, nrep, 4):
            ax = axarr[0, n]
            ax.plot(d_mg_op[i, :, 0] / 1000, (d_mg_op[i, :, 2:] < 0.25).sum(axis=-1), label=f'$λ_{{{i}}}$', color=plt.cm.coolwarm(i/(nrep-1)), lw=1)
            ax.set_ylabel('# Bound MG')
            ax.legend(loc='lower center', ncols=2, bbox_to_anchor=(0.5, 1), frameon=False)
            ax = axarr[1, n]
            ax.plot(d_op_mg[i, :, 0] / 1000, (d_op_mg[i, :, 2:] < 0.25).sum(axis=-1), label=f'$λ_{{{i}}}$', color=plt.cm.coolwarm(i/(nrep-1)), lw=1)
            ax.set_ylabel('# Bound OP')
            ax = axarr[2, n]
            ax.scatter(cv[i, ::10, 0] / 1000, cv[i, ::10, -1], label=f'$λ_{{{i}}}$', color=plt.cm.coolwarm(i/(nrep-1)), s=1)
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_ylabel('Replica index')
            ax.set_xlabel("Time [ns]")
            ax.set_xlim(0, cv_dmx[i, :, 0].max() / 1000)
    save(fig, 'bound_mg_lambda', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    fig, ax = plt.subplots()
    for i in range(nrep):
        hist, n_edges = np.histogram((d_mg_op[i, :, 2:] < 0.28).sum(axis=-1), density=True, bins=20)
        ax.plot(edges_to_range(n_edges), hist, color=plt.cm.rainbow(i/(nrep-1)), label=f"$λ_{{{i}}}$")
    ax.legend()
    ax.set_xlabel(r'$n_\mathrm{inner}$')
    save(fig, 'n_inner_dist', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    fig, ax = plt.subplots()
    for i in range(nrep):
        hist, n_edges = np.histogram(((d_mg_op[i, :, 2:] > 0.28)*(d_mg_op[i, :, 2:] < 0.5)).sum(axis=-1), density=True, bins=10)
        ax.plot(edges_to_range(n_edges), hist, color=plt.cm.rainbow(i/(nrep-1)), label=f"$λ_{{{i}}}$")
    ax.legend()
    ax.set_xlabel(r'$n_\mathrm{outer}$')
    save(fig, 'n_outer_dist', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    fig, ax = plt.subplots()
    for i in range(nrep):
        hist, n_edges = np.histogram((d_mg_op[i, :, 2:] < 0.42).sum(axis=-1), density=True, bins=10)
        ax.plot(edges_to_range(n_edges), hist, color=plt.cm.rainbow(i/(nrep-1)), label=f"$λ_{{{i}}}$")
    ax.legend()
    ax.set_xlabel('$n_{d<4.2~Å}$')
    save(fig, 'n_tot_dist', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    # Composition of each thermodynamic state
    fig, ax = plt.subplots()
    bottom = np.zeros(nrep)
    n_frames = cv_dmx.shape[1]
    for k in range(nrep):
        x, counts = np.unique(cv_dmx[k, :, -2].astype(int), return_counts=True)
        ax.bar(x, counts/n_frames*100, bottom=bottom[x], color=plt.cm.rainbow(k/(nrep-1)), label=f'{k}', edgecolor='black')
        bottom[x] += counts/n_frames*100
    ax.legend(title='Replica id.', loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, reverse=True, labelspacing=1)
    ax.set_xlabel("Thermodynamic state")
    ax.set_xticks(range(nrep), [f"$λ_{{{k}}}$" for k in range(nrep)])
    ax.set_ylabel("Frames [%]")
    ax.set_ylim(0, 100)
    ax.spines[['left', 'right']].set_visible(False)
    save(fig, 'replica_occupancy', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    # Composition of each replica
    fig, ax = plt.subplots()
    bottom = np.zeros(nrep)
    n_frames = cv_dmx.shape[1]
    for k in range(nrep):
        x, counts = np.unique(cv[k, :, -1].astype(int), return_counts=True)
        ax.bar(x, counts/n_frames*100, bottom=bottom[x], color=plt.cm.coolwarm(k/(nrep-1)), label=f'$λ_{{{k}}}$', edgecolor='black')
        bottom[x] += counts/n_frames*100
    ax.legend(title='State', loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, reverse=True, labelspacing=1)
    ax.set_xlabel("Replica index")
    ax.set_xticks(range(nrep))
    ax.set_ylabel("Frames [%]")
    ax.set_ylim(0, 100)
    ax.spines[['left', 'right']].set_visible(False)
    save(fig, 'lambda_occupancy', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    # Bound Mg²⁺ configurations along each replica
    figs = []
    for k in range(nrep):
        fig, axarr = plt.subplots(nrows=2, height_ratios=[0.9, 0.1], sharex=True)
        is_bound_mg = (nox_dmx[k, :, 1:-2:3] > 0.5) * 1 + (nox_dmx[k, :, 1:-2:3] > 1.5) * 1 + (nox_dmx[k, :, 1:-2:3] > 2.5) * 1
        cmap = plt.cm.gnuplot2_r
        custom_patches = [Patch(color=cmap(0/3), label='Unbound'),
                          Patch(color=cmap(1/3), label='Monodentate'),
                          Patch(color=cmap(2/3), label='Bidentate'),
                          Patch(color=cmap(3/3), label='> Bidentate')]
        ax = axarr[0]
        ax.imshow(is_bound_mg.T, origin='lower', extent=[0, len(is_bound_mg) / 10, 0, is_bound_mg.shape[1]], aspect='auto', cmap=cmap, interpolation='none')
        ax.set_ylabel(r"Mg$^{2+}$ index")
        ax.set_title(f"Replica {k}")
        ax.legend(handles=custom_patches)
        ax = axarr[1]
        ax.imshow(cv_dmx[k, :, [-2]], cmap=plt.cm.coolwarm, vmin=0, vmax=nrep-1, origin='lower', extent=[*(cv_dmx[k, [0, -1], 0] / 1000), 0, 1], aspect='auto')
        ax.set_xlabel("Time [ns]")
        figs.append(fig)
    save(figs, 'heatmap_mg_index', system, rep)

In [ ]:
for system, rep in PAIRS:
    cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]
    figs = []
    for n in range(4):
        fig, axarr = plt.subplots(nrows=3, sharex=True, figsize=(4, 6))
        fig.suptitle(f"{system} {ff} replicate {rep}")
        for i in range(n, nrep, 4):
            ax = axarr[0]
            is_mono = (nox[i, :, 1:-2:3] < 1.5) * (nox[i, :, 1:-2:3] > 0.5)
            is_bi = (nox[i, :, 1:-2:3] > 1.5) * (nox[i, :, 1:-2:3] < 2.5)
            ax.plot(nox[i, :, 0] / 1000, is_mono.sum(axis=-1), color=plt.cm.coolwarm(i/(nrep-1)), label=f'$λ_{{{i}}}$')
            ax.set_ylabel('# Bound MG monodentate')
            ax = axarr[1]
            ax.plot(nox[i, :, 0] / 1000, is_bi.sum(axis=-1), color=plt.cm.coolwarm(i/(nrep-1)), label=f'$λ_{{{i}}}$')
            ax.set_ylabel('# Bound MG bidentate')
            ax.legend(loc='center', bbox_to_anchor=(1, 0.5))
            ax = axarr[2]
            ax.plot(cv[i, :, 0] / 1000, cv[i, :, -1], color=plt.cm.coolwarm(i/(nrep-1)), label=f'$λ_{{{i}}}$')
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_ylabel('λ index')
            ax.set_xlabel("Time [ns]")
        figs.append(fig)
    save(figs, 'bound_mg_denticity', system, rep)

## Diagnostics

Displayed only, for one system and replicate.

In [ ]:
system, rep = PAIRS[0]
cv_dmx, cv, nox_dmx, nox, d_mg_op_dmx, d_op_mg_dmx, d_mg_op, d_op_mg = data[system, rep]

In [ ]:
# Phosphate oxygens bound along two replicas
for k in [0, 3]:
    fig, ax = plt.subplots(figsize=(6, 8))
    # O1P or O2P
    is_bound_op = np.logical_or(d_op_mg_dmx[k, :, 2::2] < 0.25, d_op_mg_dmx[k, :, 3::2] < 0.25) * 1 + np.logical_or(d_op_mg_dmx[k, :, 2::2] < 0.42, d_op_mg_dmx[k, :, 3::2] < 0.42) * 1
    cmap = plt.cm.OrRd
    custom_patches = [Patch(color=cmap(0.), label='Unbound'),
                      Patch(color=cmap(.5), label='Outer-bound'),
                      Patch(color=cmap(1.), label='Inner-bound')]
    ax.imshow(is_bound_op.T, origin='lower', extent=[0, len(is_bound_op)/10, 0, is_bound_op.shape[1]], aspect='auto', cmap=cmap, interpolation='none')
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel(r"Residue index")
    ax.set_title(f"Replica {k}")
    ax.legend(handles=custom_patches)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(nrep):
    ρ_d, bins = np.histogram(d_mg_op[i, :, 2:].flatten(), bins=100, density=True)
    d_range = edges_to_range(bins)
    ax.plot(d_range, ρ_d, label=f'$λ_{{{i}}}$', color=plt.cm.cool(i/(nrep-1)))
ax.legend(loc='right')
ax.set_xlabel(r'$d_\mathrm{min}$')
ax.set_ylabel(r'$ρ(d_\mathrm{min})$');

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(nrep):
    ρ_d, bins = np.histogram(nox[i, :, 1:-2:3].flatten(), bins=100, density=True)
    d_range = edges_to_range(bins)
    ax.plot(d_range, ρ_d, label=f'$λ_{{{i}}}$', color=plt.cm.cool(i/(nrep-1)))
ax.legend()
ax.set_xlabel(r'$n_\mathrm{O_P}$')
ax.set_ylabel(r'$ρ(n_\mathrm{O_P})$')
ax.set_ylim(0, 1);

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(nrep):
    ρ_d, bins = np.histogram(nox[i, :, 2:-2:3].flatten(), bins=100, density=True)
    d_range = edges_to_range(bins)
    ax.plot(d_range, ρ_d, label=f'$λ_{{{i}}}$', color=plt.cm.cool(i/(nrep-1)))
ax.legend()
ax.set_xlabel(r'$n_\mathrm{O_W}$')
ax.set_ylabel(r'$ρ(n_\mathrm{O_W})$');

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(nrep):
    ρ_d, bins = np.histogram(nox[i, :, 1:-2:3].flatten() - nox[i, :, 2:-2:3].flatten() + 6, bins=100, density=True)
    d_range = edges_to_range(bins)
    ax.plot(d_range, ρ_d, label=f'$λ_{{{i}}}$', color=plt.cm.cool(i/(nrep-1)))
ax.legend()
ax.set_xlabel(r'$Δ(n_\mathrm{O_P} - n_\mathrm{O_W})$')
ax.set_ylabel(r'$ρ(Δ(n_\mathrm{O_P} - n_\mathrm{O_W}))$');

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(nrep):
    ρ_d, bins = np.histogram(nox[i, :, 1:-2:3].flatten() - nox[i, :, 2:-2:3].flatten() + 6, bins=100, density=True)
    d_range = edges_to_range(bins)
    ax.plot(d_range, -RT*np.log(ρ_d), label=f'$λ_{{{i}}}$', color=plt.cm.cool(i/(nrep-1)))
ax.legend()
ax.set_xlabel(r'$Δ(n_\mathrm{O_P} - n_\mathrm{O_W})$')
ax.set_ylabel(r'$-RT\ln ρ(Δ(n_\mathrm{O_P} - n_\mathrm{O_W}))$');

In [ ]:
fig, axarr = plt.subplots(ncols=4, nrows=2, figsize=(12, 6), sharex=True, sharey=True)
norm = Normalize()
for i, ax in enumerate(axarr.flatten()):
    ρ_d, x_edges, y_edges = np.histogram2d(nox[i, :, 1:-2:3].flatten(), nox[i, :, 2:-2:3].flatten(), bins=200, density=True)
    x_range, y_range = edges_to_range(x_edges), edges_to_range(y_edges)
    cf = ax.contourf(x_range, y_range, -RT*np.log(ρ_d.T), cmap='magma_r', norm=norm)
    ax.set_title(f'$λ_{{{i}}}$')
    ax.set_xlabel(r'$n_\mathrm{O_P}$')
    ax.set_ylabel(r'$n_\mathrm{O_W}$')
    plt.colorbar(cf)

In [ ]:
# Bias distribution as a function of λ, including λ_0
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(nrep):
    ρ_u, bins = np.histogram(cv[i, :, 1], bins=50, density=True)
    u_range = edges_to_range(bins)
    ax.plot(u_range, ρ_u, label=f'$λ_{{{i}}}$', color=plt.cm.cool(i/(nrep-1)))
ax.legend()
ax.set_xlabel(r'$U_\mathrm{bias}$ [kJ/mol]')
ax.set_ylabel(r'$ρ(U_\mathrm{bias})$');